# 📘 Module 04 · Notebook 04 — Evaluasi LLM (UKUR)

"Kelihatannya bagus" bukan bukti. Di sini kita **mengukur** kualitas output LLM dengan metrik
objektif, lalu **menguji apakah perbedaan dua model benar-benar signifikan** (bukan kebetulan).

Kita bandingkan **Qwen2.5-1.5B (A)** vs **Qwen2.5-3B (B)** pada tugas terjemahan ID→EN.

## Apa yang akan kita pelajari?
1. Metrik *referenced*: ROUGE, BLEU, BERTScore
2. Perplexity (kelancaran output)
3. **Uji signifikansi statistik** (paired t-test, Cohen's d, 95% CI)
4. **LLM-as-judge** lokal (Qwen) & cloud besar (NVIDIA NIM / Gemini)

> Harness ini dipakai lagi di **notebook 03** untuk membuktikan fine-tuning benar memperbaiki model.

In [1]:
# Pin transformers<5 (konsisten dgn modul); + library metrik + openai (klien utk NVIDIA NIM/cloud)
!pip install -q "transformers>=4.44,<5" accelerate evaluate rouge_score sacrebleu bert_score openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.6 MB/s eta 0:00:00


In [2]:
import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def gpu_mem(tag=""):
    if not torch.cuda.is_available():
        return
    print(f"[{tag}] allocated={torch.cuda.memory_allocated()/1e9:.2f} GB | reserved={torch.cuda.memory_reserved()/1e9:.2f} GB")

device: cuda
GPU: Tesla T4


## 1. Data: Terjemahan ID→EN dengan Referensi

Metrik *referenced* membutuhkan **jawaban acuan** (referensi). Kita siapkan pasangan kalimat
Indonesia + terjemahan Inggris yang benar, lalu minta model menerjemahkan dan membandingkan
hasilnya dengan referensi.

In [3]:
sources = [
    "Kecerdasan buatan akan mengubah banyak pekerjaan di masa depan.",
    "Saya suka membaca buku di pagi hari sambil minum kopi.",
    "Pemerintah mengumumkan kebijakan baru tentang energi terbarukan.",
    "Tim sepak bola itu memenangkan pertandingan dengan skor tiga kosong.",
    "Belajar bahasa asing membutuhkan latihan yang konsisten setiap hari.",
    "Harga bahan pangan naik tajam menjelang hari raya.",
    "Dokter menyarankan pasien untuk lebih banyak berolahraga.",
    "Aplikasi ini membantu pengguna mengatur keuangan pribadi mereka.",
    "Hujan deras menyebabkan banjir di beberapa wilayah kota.",
    "Perusahaan teknologi itu merilis produk terbarunya bulan lalu.",
]
references = [
    "Artificial intelligence will change many jobs in the future.",
    "I like reading books in the morning while drinking coffee.",
    "The government announced a new policy on renewable energy.",
    "That football team won the match with a score of three to nil.",
    "Learning a foreign language requires consistent practice every day.",
    "Food prices rose sharply ahead of the holiday.",
    "The doctor advised the patient to exercise more.",
    "This application helps users manage their personal finances.",
    "Heavy rain caused flooding in several areas of the city.",
    "That technology company released its newest product last month.",
]
print(f"{len(sources)} pasang kalimat ID->EN dengan terjemahan referensi.")

10 pasang kalimat ID->EN dengan terjemahan referensi.


## 2. Generate dengan Dua Model

Kita terjemahkan semua kalimat dengan **A = Qwen2.5-1.5B** lalu **B = Qwen2.5-3B**. Model dimuat
**bergantian** agar hemat VRAM (A dibebaskan sebelum B dimuat). Model B disimpan untuk dipakai
sebagai **juri lokal** di bagian akhir.

In [4]:
def translate_all(model_name, sources, keep=False):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto")
    preds = []
    for src in sources:
        messages = [
            {"role": "system", "content": "You are a precise translator. Translate the Indonesian text into natural English. Output ONLY the English translation."},
            {"role": "user", "content": src},
        ]
        inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(mdl.device)
        out = mdl.generate(**inputs, max_new_tokens=80, do_sample=False,
                           pad_token_id=tok.eos_token_id)
        preds.append(tok.decode(out[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip())
    if keep:
        return preds, tok, mdl
    del mdl
    torch.cuda.empty_cache()
    return preds, None, None

print("Menerjemahkan dengan Qwen2.5-1.5B (model A)...")
preds_A, _, _ = translate_all("Qwen/Qwen2.5-1.5B-Instruct", sources)
gpu_mem("A dibebaskan")
print("Menerjemahkan dengan Qwen2.5-3B (model B, disimpan utk juri)...")
preds_B, judge_tok, judge_model = translate_all("Qwen/Qwen2.5-3B-Instruct", sources, keep=True)
gpu_mem("B resident")
for s, a, b in zip(sources, preds_A, preds_B):
    print(f"\nID: {s}\n  A(1.5B): {a}\n  B(3B) : {b}")

Menerjemahkan dengan Qwen2.5-1.5B (model A)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[B resident] allocated=6.18 GB | reserved=6.31 GB

ID: Kecerdasan buatan akan mengubah banyak pekerjaan di masa depan.
  A(1.5B): Artificial intelligence will change many jobs in the future.
  B(3B) : Artificial intelligence will transform many jobs in the future.

ID: Saya suka membaca buku di pagi hari sambil minum kopi.
  A(1.5B): I enjoy reading books in the morning while drinking coffee.
  B(3B) : I like reading books in the morning while drinking coffee.

ID: Pemerintah mengumumkan kebijakan baru tentang energi terbarukan.
  A(1.5B): The government has announced new policies regarding renewable energy.
  B(3B) : The government announces new policies regarding renewable energy.

ID: Tim sepak bola itu memenangkan pertandingan dengan skor tiga kosong.
  A(1.5B): The soccer team won the game with a score of three goals down.
  B(3B) : The soccer team won the match with a 3–0 score.

ID: Belajar bahasa asing membutuhkan latihan yang konsisten setiap hari.
  A(1.5B): Learning a foreig

## 3. Metrik Referenced: ROUGE, BLEU, BERTScore

- **ROUGE** — overlap n-gram (recall-oriented), populer untuk ringkasan.
- **BLEU** — presisi n-gram, standar untuk terjemahan.
- **BERTScore** — kemiripan **makna** via embedding (menangkap parafrase yang ROUGE/BLEU lewatkan).

Makin tinggi = makin dekat ke referensi.

In [5]:
import evaluate
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")
bertscore = evaluate.load("bertscore")

def referenced_scores(preds):
    r = rouge.compute(predictions=preds, references=references)
    b = bleu.compute(predictions=preds, references=[[x] for x in references])
    bs = bertscore.compute(predictions=preds, references=references, lang="en")
    return r["rougeL"], b["score"], np.array(bs["f1"])

rougeL_A, bleu_A, bs_A_f1 = referenced_scores(preds_A)
rougeL_B, bleu_B, bs_B_f1 = referenced_scores(preds_B)

print(f"{'metrik':<14}{'A (1.5B)':>12}{'B (3B)':>12}")
print(f"{'ROUGE-L':<14}{rougeL_A:>12.3f}{rougeL_B:>12.3f}")
print(f"{'BLEU':<14}{bleu_A:>12.2f}{bleu_B:>12.2f}")
print(f"{'BERTScore-F1':<14}{bs_A_f1.mean():>12.3f}{bs_B_f1.mean():>12.3f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


metrik            A (1.5B)      B (3B)
ROUGE-L              0.775       0.790
BLEU                 56.81       60.76
BERTScore-F1         0.982       0.986


## 4. Perplexity (Kelancaran Output)

Perplexity mengukur seberapa "wajar" sebuah teks menurut model bahasa. Kita ukur kelancaran
terjemahan Inggris masing-masing model memakai **gpt2** sebagai penilai. **Makin RENDAH = makin
lancar** (reference-free — tak butuh jawaban acuan).

In [6]:
perplexity = evaluate.load("perplexity", module_type="metric")
ppl_A = perplexity.compute(predictions=preds_A, model_id="gpt2")["mean_perplexity"]
ppl_B = perplexity.compute(predictions=preds_B, model_id="gpt2")["mean_perplexity"]
print("Perplexity output (gpt2; makin RENDAH makin lancar):")
print(f"  A (1.5B): {ppl_A:.1f}")
print(f"  B (3B) : {ppl_B:.1f}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Perplexity output (gpt2; makin RENDAH makin lancar):
  A (1.5B): 69.8
  B (3B) : 54.4


## 5. Uji Signifikansi Statistik

B terlihat lebih tinggi — tapi **apakah nyata atau kebetulan?** Kita pakai **paired t-test** atas
skor BERTScore-F1 per-sampel, plus **Cohen's d** (ukuran efek) dan **95% CI** dari selisih.
`p < 0.05` → perbedaan signifikan secara statistik.

In [7]:
from scipy import stats

t_stat, p_val = stats.ttest_rel(bs_B_f1, bs_A_f1)   # paired: B vs A
diff = bs_B_f1 - bs_A_f1
cohens_d = diff.mean() / diff.std(ddof=1)
ci = stats.t.interval(0.95, len(diff) - 1, loc=diff.mean(), scale=stats.sem(diff))

print(f"BERTScore-F1 per-sampel (n={len(diff)})")
print(f"  mean A={bs_A_f1.mean():.3f} | mean B={bs_B_f1.mean():.3f}")
print(f"  paired t-test: t={t_stat:.3f}, p={p_val:.4f}")
print(f"  -> {'SIGNIFIKAN' if p_val < 0.05 else 'tidak signifikan'} pada alpha=0.05")
print(f"  Cohen's d = {cohens_d:.3f} | 95% CI selisih (B-A) = ({ci[0]:.3f}, {ci[1]:.3f})")

BERTScore-F1 per-sampel (n=10)
  mean A=0.982 | mean B=0.986
  paired t-test: t=1.647, p=0.1339
  -> tidak signifikan pada alpha=0.05
  Cohen's d = 0.521 | 95% CI selisih (B-A) = (-0.001, 0.008)


## 6. LLM-as-Judge (lokal & cloud)

Teknik modern: pakai LLM lain sebagai **juri** menilai output (reference-free, fleksibel).
Kita coba **dua** juri:

- **Juri lokal — Qwen2.5-3B** (gratis, offline). ⚠️ Ia menilai output-nya sendiri (model B) →
  cenderung **self-preference bias**.
- **Juri cloud BESAR — opsional** lewat **NVIDIA NIM** (`build.nvidia.com`, daftar gratis) atau
  Gemini API. Model jauh lebih besar **dan independen** → lebih tepercaya.

> 💡 Di Navasena, **boleh memakai API cloud** untuk model lebih besar (endpoint *OpenAI-compatible*).
> **Saat generate key NVIDIA**, di *Services Included* centang **"Public API Endpoints"** (selain
> "NGC Catalog"). Tanpa itu, call dari luar playground kena **403 Authorization failed**.

In [8]:
import re

def judge_local(src, hypothesis):
    prompt = f"""Rate this Indonesian-to-English translation from 1 to 5 (5 = perfect, natural, accurate).
Indonesian: {src}
English: {hypothesis}
Reply with ONLY a single integer from 1 to 5."""
    messages = [{"role": "user", "content": prompt}]
    inputs = judge_tok.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(judge_model.device)
    out = judge_model.generate(**inputs, max_new_tokens=4, do_sample=False,
                               pad_token_id=judge_tok.eos_token_id)
    text = judge_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"[1-5]", text)
    return int(m.group()) if m else None

ok = lambda xs: [x for x in xs if x is not None]
local_A = ok([judge_local(s, h) for s, h in zip(sources, preds_A)])
local_B = ok([judge_local(s, h) for s, h in zip(sources, preds_B)])
print("Juri LOKAL (Qwen2.5-3B) — skor rata-rata 1-5 (awas self-bias):")
print(f"  A (1.5B): {np.mean(local_A):.2f}")
print(f"  B (3B) : {np.mean(local_B):.2f}")

del judge_model
torch.cuda.empty_cache()

Juri LOKAL (Qwen2.5-3B) — skor rata-rata 1-5 (awas self-bias):
  A (1.5B): 4.70
  B (3B) : 5.00


In [9]:
# (Opsional) Juri model BESAR via cloud NVIDIA NIM — independen, mengurangi bias self-preference.
# Daftar GRATIS di https://build.nvidia.com -> Generate Personal Key.
#   PENTING: di "Services Included" centang BOTH "NGC Catalog" + "Public API Endpoints".
# Simpan di Colab Secrets (ikon kunci) sbg NVIDIA_API_KEY, aktifkan "Notebook access".
import re

NVIDIA_API_KEY = None
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    import os
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    print("NVIDIA_API_KEY tidak ada -> lewati juri cloud. (Daftar gratis di build.nvidia.com.)")
else:
    from openai import OpenAI, PermissionDeniedError, AuthenticationError
    NVIDIA_API_KEY = NVIDIA_API_KEY.strip()   # buang spasi/newline (penyebab 403 paling umum dari Colab Secrets)
    print("key:", "len =", len(NVIDIA_API_KEY), "| format nvapi- :", NVIDIA_API_KEY.startswith("nvapi-"))
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

    JUDGE_MODEL = "meta/llama-3.3-70b-instruct"   # alternatif cepat: "meta/llama-3.1-8b-instruct"

    # Cek auth dulu: kalau models.list() 403 -> masalah KEY/permission, BUKAN model.
    available = None
    try:
        available = {m.id for m in client.models.list().data}
        print(f"auth OK — {len(available)} model tersedia.")
        if JUDGE_MODEL not in available:
            JUDGE_MODEL = "meta/llama-3.1-8b-instruct"
    except (PermissionDeniedError, AuthenticationError) as e:
        print(f"\n403/401 saat auth: {e}")
        print(">>> Ini masalah KEY/PERMISSION, bukan model. Perbaiki berurutan:")
        print("    1. Pastikan key diawali 'nvapi-' & tanpa spasi (lihat baris di atas).")
        print("    2. REGENERATE key di build.nvidia.com -> centang 'Public API Endpoints' di Services Included.")
        print("       (Permission ini penyebab #1 error 403 ini di 2026.)")
        print("    3. Cek org switcher (kanan atas) memakai org yang punya permission tsb.")

    if available is not None:
        def judge_cloud(src, hyp):
            prompt = (f"Rate this Indonesian-to-English translation from 1 to 5 "
                      f"(5 = perfect, natural, accurate). Reply with ONLY a single integer 1-5.\n"
                      f"Indonesian: {src}\nEnglish: {hyp}")
            r = client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0, max_tokens=4,
                messages=[{"role": "user", "content": prompt}])
            m = re.search(r"[1-5]", r.choices[0].message.content)
            return int(m.group()) if m else None

        ok = lambda xs: [x for x in xs if x is not None]
        cloud_A = ok([judge_cloud(s, h) for s, h in zip(sources, preds_A)])
        cloud_B = ok([judge_cloud(s, h) for s, h in zip(sources, preds_B)])
        print(f"\nJuri CLOUD ({JUDGE_MODEL} via NVIDIA NIM) — independen:")
        print(f"  A (1.5B): {np.mean(cloud_A):.2f}")
        print(f"  B (3B) : {np.mean(cloud_B):.2f}")

key: len = 70 | format nvapi- : True
auth OK — 120 model tersedia.

Juri CLOUD (meta/llama-3.3-70b-instruct via NVIDIA NIM) — independen:
  A (1.5B): 4.70
  B (3B) : 5.00


## Ringkasan & Jembatan

| Cara ukur | Butuh referensi? | Mengukur |
|-----------|------------------|----------|
| ROUGE / BLEU | ya | overlap n-gram (bentuk) |
| BERTScore | ya | kemiripan makna |
| Perplexity | tidak | kelancaran |
| Uji statistik | — | apakah selisih nyata (bukan noise) |
| LLM-as-judge (lokal/cloud) | tidak | kualitas via juri (lokal: awas bias; cloud: besar & independen) |

Disiplin "**ukur, jangan kira-kira**" ini akan kita pakai di **notebook 03** untuk membuktikan
model **fine-tuned** benar-benar mengalahkan base. Untuk evaluasi **RAG** (faithfulness, context
precision) → **RAGAS** di **Module 05** (juga bisa pakai juri cloud).